# Report Figures

Builds the two benchmark figures used in the report from the saved summary tables of the full development run (`stability_across_splits_descriptive.csv` and `rankings_matched_descriptive.csv`). No attack or detection is rerun.

- `fig_asr_vs_edits_report`: mean ASR versus actual changed-word ratio (support 1, ten splits, error bars = SD across splits). Matched-budget points are dropped once the budget saturates. MarkLLM uses a word-edit proxy, and the hybrid is not budget-matched to a random baseline, so this comparison is descriptive.
- `fig_auc_errorbars_report`: normalized area under the ASR versus actual edit ratio curve (mean ± SD across ten splits).

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESULTS = Path("../../../results/attacks/fair_comparison_benchmark_v2/development_full")
OUT = RESULTS
SUPPORT = 1

INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#ffffff"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE})

# ---------- Figure 2: ASR vs actual edit ratio ----------
stab = pd.read_csv(RESULTS / "stability_across_splits_descriptive.csv")
stab = stab[stab.support == SUPPORT]
SERIES = [  # (method, label, colour, linestyle, marker, label offset, label alignment)
    ("markllm_original_native", "MarkLLM WordNet (edit proxy)", "#2a78d6", "--", "o", (6, 0), "left"),
    ("random_wordnet_matched",  "Random WordNet",               "#eb6834", "-",  "s", None, None),
    ("v2_rank_matched",         "V2 ranked word bigrams",       "#1baf7a", "-",  "^", None, None),
    ("v3_hybrid_native",        "V3 hybrid (V3→V2→V1)",         "#eda100", "-",  "D", (6, 0), "left"),
]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for method, label, colour, ls, marker, offset, ha in SERIES:
    d = stab[stab.method == method].sort_values("ratio")
    keep, last = [], -1.0
    for _, r in d.iterrows():          # drop points where the matched budget has saturated
        if r.mean_actual_edit_ratio * 100 > last + 0.5:
            keep.append(r); last = r.mean_actual_edit_ratio * 100
    d = pd.DataFrame(keep)
    x, y, e = d.mean_actual_edit_ratio * 100, d.mean_asr * 100, d.sd_asr * 100
    ax.errorbar(x, y, yerr=e, color=colour, ls=ls, lw=2, marker=marker, ms=6,
                mec=SURFACE, mew=1.2, capsize=2.5, elinewidth=1, label=label)
    if offset is not None:  # direct labels only where they do not collide; the legend names all series
        ax.annotate(label, (x.iloc[-1], y.iloc[-1]), xytext=offset, textcoords="offset points",
                    va="center", ha=ha, fontsize=8, color=INK)
ax.set_xlabel("Actual changed-word ratio (%)")
ax.set_ylabel("ASR on initially detected texts (%)")
ax.set_xlim(0, 62); ax.set_ylim(0, 100)
ax.grid(True, color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "fig_asr_vs_edits_report.png", dpi=300)
fig.savefig(OUT / "fig_asr_vs_edits_report.pdf")

# ---------- Figure 1 (updated): normalized AUC with error bars ----------
rk = pd.read_csv(RESULTS / "rankings_matched_descriptive.csv")
rk = rk[(rk.support == SUPPORT) & (rk.comparison_group == "main_matched")]
NAMES = {"v2_synonym_guided_matched": "V2 · guided synonym", "v2_rank_matched": "V2 · word bigram",
         "v3_rank_matched": "V3 · token bigram", "v2_pool_random_matched": "V2 · random pool",
         "v3_pool_random_matched": "V3 · random pool", "v1_rank_matched": "V1 · word unigram",
         "random_wordnet_matched": "Random WordNet"}
rk = rk.assign(name=rk.method.map(NAMES)).sort_values("mean_normalized_auc")
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ax.barh(rk.name, rk.mean_normalized_auc, xerr=rk.sd_across_splits, color="#2a78d6", height=0.6,
        error_kw={"ecolor": INK2, "elinewidth": 1, "capsize": 2.5})
for i, (v, s) in enumerate(zip(rk.mean_normalized_auc, rk.sd_across_splits)):
    ax.text(v + s + 0.003, i, f"{v:.3f}", va="center", fontsize=8, color=INK)
ax.set_xlabel("Normalized area under ASR vs. actual edit ratio (mean ± SD, 10 splits)")
ax.set_xlim(0, 0.2)
ax.grid(True, axis="x", color=GRID, lw=0.6); ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(OUT / "fig_auc_errorbars_report.png", dpi=300)
fig.savefig(OUT / "fig_auc_errorbars_report.pdf")
print("Saved figures to", OUT)


Saved figures to ../../../results/attacks/fair_comparison_benchmark_v2/development_full
